# Prepare Spectrograms
Original workflow: `GPN_UAV_data/extract&draw_picture.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
!pip install --user pyarrow pillow

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import os
import numpy as np

def transpose_npy_files_to_new_path(root_folder, output_folder):
    """
    遍历指定根文件夹 (root_folder) 及其所有子文件夹，将找到的每个 .npy 数组进行转置 (transpose)，
    并保存到目标文件夹 (output_folder) 中，保持原有的目录结构不变。

    Args:
        root_folder (str): 要开始搜索的原始根文件夹路径 (源路径 A)。
        output_folder (str): 用于保存转置后文件的新根文件夹路径 (目标路径 B)。
    """
    if not os.path.isdir(root_folder):
        print(f"错误: 原始文件夹路径 '{root_folder}' 不存在或不是一个目录。")
        return
        
    # 确保目标输出文件夹存在
    os.makedirs(output_folder, exist_ok=True)
    print(f"开始在 '{root_folder}' 中搜索 .npy 文件，并将转置结果保存到 '{output_folder}'...")
    
    # os.walk() 会递归地遍历目录树
    for dirpath, dirnames, filenames in os.walk(root_folder):
        # 计算当前子文件夹相对于根文件夹的相对路径
        relative_path = os.path.relpath(dirpath, root_folder)
        
        # 构建目标路径中的对应子文件夹路径
        output_dirpath = os.path.join(output_folder, relative_path)
        
        # 如果目标子文件夹不存在，则创建它 (保持结构不变的关键步骤)
        os.makedirs(output_dirpath, exist_ok=True)
        
        for filename in filenames:
            # 检查文件是否是以 .npy 结尾
            if filename.endswith(_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell1.transpose_npy_files_to_new_path.path')):
                npy_file_path = os.path.join(dirpath, filename)         # 原始文件路径
                output_file_path = os.path.join(output_dirpath, filename) # 目标保存路径
                
                try:
                    # <configured>. 加载 .npy 文件
                    data = np.load(npy_file_path)
                    
                    # <configured>. 检查数据维度
                    if data.ndim < 2:
                        print(f"跳过文件: {npy_file_path}，因为维度小于2 (ndim={data.ndim})，无法进行转置。")
                        continue

                    # <configured>. 执行转置操作
                    transposed_data = data.T 
                    
                    # <configured>. **保存转置后的数据到新路径**
                    np.save(output_file_path, transposed_data)
                    
                    # print(f"成功转置: {npy_file_path}")
                    # print(f" -> 保存到: {output_file_path} (原形状: {data.shape} -> 新形状: {transposed_data.shape})")
                    
                except Exception as e:
                    print(f"处理文件时发生错误: {npy_file_path}。错误信息: {e}")

# ====================================================================
# 使用示例
# ====================================================================

# 原始数据根文件夹 (A 路径)
source_folder = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell1.module.source_folder') 

# 转置后数据的新根文件夹 (B 路径)
# 建议使用一个不同的、明确的新文件夹，例如在源文件夹同级创建一个 'npy_transposed'
output_folder = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell1.module.output_folder')

# 运行函数
transpose_npy_files_to_new_path(source_folder, output_folder) # 运行前请取消注释

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import numpy as np
import os
file_path = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell2.module.file_path')
data_array = np.load(file_path, allow_pickle=True)

print(f"--- 文件: {os.path.basename(file_path)} 的数组结构 ---")
print(f"1. 数组形状 (Shape): {data_array.shape}")
print(f"2. 数据类型 (Dtype): {data_array.dtype}")

# 额外信息：查看数组的维度
print(f"3. 数组维度 (ndim): {data_array.ndim}")

# 额外信息：打印前几个元素（如果不是很大的数组）
if data_array.ndim >= 2:
    print(f"4. 左上角 9x9 元素:\n{data_array[:9, :9]}")

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#从 .npy 文件加载二维数组数据（频谱图），并将其绘制成图像。
import numpy as np
import matplotlib.pyplot as plt

def plot_spectrogram_from_npy(npy_filepath, title="Spectrogram Plot"):
    """
    Args:
        npy_filepath (str): .npy 文件的完整路径。
        title (str): 图像的标题。
    """
    try:
        # <configured>. 加载数据
        # 假设 .npy 文件存储的是一个二维 NumPy 数组
        spectrogram_data = np.load(npy_filepath)
        
        # 检查数据是否为二维
        if spectrogram_data.ndim != 2:
            print(f"错误: 加载的数据维度不是 2D，而是 {spectrogram_data.ndim}D。无法绘制频谱图。")
            return

        print(f"成功加载文件: {npy_filepath}")
        print(f"数据形状 (时间点 x 频率点): {spectrogram_data.shape}")

        # <configured>. 绘制图像
        plt.figure(figsize=(10, 6)) # 设置图的大小 (可选)
        
        # 使用 imshow 或 pcolormesh 绘制颜色图。
        # 'origin'='lower' 使频率轴从底部开始（通常更符合频谱图习惯）。
        # 'aspect'='auto' 允许图像填充整个画布。
        
        # vmin 和 vmax 可以用来设置颜色映射的范围，以获得更好的对比度
        plt.imshow(spectrogram_data, 
                   aspect='auto', 
                   origin='lower', 
                   cmap='viridis') # 'viridis' 是一个常用的颜色映射

        # 添加颜色条（Colorbar）来显示颜色与值（强度/dB）的对应关系
        plt.colorbar(label='Intensity (e.g., dB)')

        # 设置轴标签和标题
        plt.title(title)
        plt.xlabel("Time Bins")
        plt.ylabel("Frequency Bins")
        
        # <configured>. 显示图像
        plt.show()
        
    except FileNotFoundError:
        print(f"错误: 找不到文件 {npy_filepath}")
    except Exception as e:
        print(f"绘制过程中发生错误: {e}")

# Historical configuration omitted; provide values through private configuration.
FILE_PATH =_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell3.module.FILE_PATH')
if __name__ == "__main__":
    # Historical configuration omitted; provide values through private configuration.
#     try:
#         # 创建一个 <configured> x <configured> 的随机数据作为示例频谱图
# Historical configuration omitted; provide values through private configuration.
#         np.save(FILE_PATH, dummy_data)
#         print(f"已创建模拟文件: {FILE_PATH}")
#     except Exception:
#         # 如果创建失败，可能是权限问题，忽略
#         pass
    
    # --- 步骤 <configured>: 调用绘图函数 ---
    plot_spectrogram_from_npy(
        npy_filepath=FILE_PATH, 
        title="Example Spectrogram from .npy File"
    )

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import numpy as np
import matplotlib.pyplot as plt
import pywt
from typing import Tuple, Dict

def wavelet_denoise_comparison(
    image_array: np.ndarray, 
    wavelet_name: str = None, 
    level: int = 3, 
    threshold_mode: str = None,
    title: str = "DWT 降噪效果对比"
) -> Dict[str, np.ndarray]:
    """
    对输入的二维图像（频谱图）进行小波降噪，并返回用于对比的关键图像。
    
    Args:
        image_array: 输入的二维 NumPy 图像数组（如频谱图）。
        wavelet_name: 使用的小波基名称（如 'db4', 'haar'）。
        level: 分解的级数。
        threshold_mode: 阈值处理模式 ('soft' 或 'hard')。
        title: 绘图的标题。

    Returns:
        包含原始、去噪和噪声图像的字典。
    """
    
    wavelet_name = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell4.wavelet_denoise_comparison.wavelet_name', wavelet_name)
    threshold_mode = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell4.wavelet_denoise_comparison.threshold_mode', threshold_mode)
    if image_array.ndim != 2:
        raise ValueError("输入数组必须是二维图像。")
        
    H, W = image_array.shape

    # --- <configured>. 小波分解和阈值计算 ---
    
    # <configured> 分解
    coeffs = pywt.wavedec2(image_array, wavelet_name, level=level)
    
    # coeffs 结构: ([cA_n, (cH_n, cV_n, cD_n)], ..., (cH_1, cV_1, cD_1))
    
    # Historical configuration omitted; provide values through private configuration.
    # 估计噪声标准差 (sigma)，通常基于最高频细节系数的中值
    # 假设噪声为高斯白噪声
    sigma_est = np.median(np.abs(coeffs[-1][0])) / 0.6745
    threshold = sigma_est * np.sqrt(2 * np.log(image_array.size))

    # --- <configured>. 阈值处理和重构 ---
    
    denoised_coeffs = list(coeffs)
    
    # Historical configuration omitted; provide values through private configuration.
    for i in range(1, len(coeffs)):
        # coeffs[i] 是一个 (cH, cV, cD) 元组
        denoised_level = []
        for detail_coeff in coeffs[i]:
            denoised_coeff = pywt.threshold(
                detail_coeff, 
                value=threshold, 
                mode=threshold_mode
            )
            denoised_level.append(denoised_coeff)
        
        denoised_coeffs[i] = tuple(denoised_level)

    # <configured> 重构信号
    denoised_image = pywt.waverec2(denoised_coeffs, wavelet_name)
    
    # <configured> 裁剪重构图像以匹配原始形状
    denoised_image = denoised_image[:H, :W]

    # --- <configured>. 计算被去除的噪声图像 ---
    
    # 噪声 = 原始图像 - 去噪图像
    noise_image = image_array - denoised_image
    
    # --- <configured>. 可视化对比 ---
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 确保所有图使用相同的颜色映射范围（vmin, vmax）
    v_min, v_max = image_array.min(), image_array.max()

    # 图像 <configured>: 原始图像
    im0 = axes[0].imshow(image_array, cmap='magma', aspect='auto', vmin=v_min, vmax=v_max)
    axes[0].set_title(f"1. 原始图像 (SNR = {np.std(image_array) / np.std(noise_image):.2f})", fontsize=14)
    axes[0].axis('off')

    # 图像 <configured>: 去噪图像
    im1 = axes[1].imshow(denoised_image, cmap='magma', aspect='auto', vmin=v_min, vmax=v_max)
    axes[1].set_title(f"2. DWT 去噪图像 ({wavelet_name}, {level}级)", fontsize=14)
    axes[1].axis('off')
    
    # 图像 <configured>: 被去除的噪声（细节）
    # 噪声图的颜色映射应以 <configured> 为中心
    noise_vmax = np.abs(noise_image).max()
    im2 = axes[2].imshow(noise_image, cmap='coolwarm', aspect='auto', vmin=-noise_vmax, vmax=noise_vmax)
    axes[2].set_title("3. 被去除的噪声（差异）", fontsize=14)
    axes[2].axis('off')

    # 添加颜色条
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    return {
        "original": image_array,
        "denoised": denoised_image,
        "noise": noise_image
    }

# --- 演示示例 ---
if __name__ == "__main__":
    # Historical configuration omitted; provide values through private configuration.
    FILE_PATH=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell4.module.FILE_PATH')

    image=np.load(FILE_PATH)
     

    # <configured>. 执行并对比 DWT 降噪效果
    print("开始进行 DWT 降噪对比...")
    results = wavelet_denoise_comparison(
        image_array=image,
        wavelet_name=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell4.module.wavelet_name'),  # 也可以尝试 'haar', 'db4'
        level=3,
        threshold_mode=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell4.module.threshold_mode'), # 软阈值（'soft'）比硬阈值（'hard'）更平滑
        title=f"频谱图 DWT 降噪对比 (SNR=dB, sym4/4级/Soft阈值)"
    )
    
    print("DWT 降噪完成。")

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import numpy as np
from typing import Dict, Tuple

def fun(a=0.4,
    b=0.7,
    data_array=np.load(_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell5.fun.path'),allow_pickle=True)):
    '\n    计算二维 NumPy 数组中数据点在给定三个区间的数量。\n    假设输入数据已经归一化，范围在 [<configured>, <configured>] 之间。\n\n    区间定义为：\n    <configured>. 范围 <configured> 到 a： [<configured>, a)    （包含 <configured>，不包含 a）\n    <configured>. 范围 a 到 b： [a, b)    （包含 a，不包含 b）\n    <configured>. 范围 b 到 <configured>： [b, <configured>]    （包含 b，包含 <configured>）\n\n    Args:\n        data_array: 待分析的二维 NumPy 数组。\n        a: 第一个分界点 (<configured> < a < b)。\n        b: 第二个分界点 (a < b < <configured>)。\n\n    Returns:\n        一个字典，包含三个区间的数据点数量。\n    '

    # 确保 a 和 b 的顺序和范围合理性
    if not (0 < a < b < 1):
        raise ValueError('参数 a 和 b 必须满足 <configured> < a < b < <configured>。')

    # <configured>. 计算 [<configured>, a) 区间的数据点数
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    mask_0a = data_array < a
    count_0a = np.count_nonzero(mask_0a)

    # <configured>. 计算 [a, b) 区间的数据点数
    # 使用布尔索引： data_array >= a 且 data_array < b
    mask_ab = (data_array >= a) & (data_array < b)
    count_ab = np.count_nonzero(mask_ab)

    # <configured>. 计算 [b, <configured>] 区间的数据点数
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    mask_b1 = data_array >= b
    count_b1 = np.count_nonzero(mask_b1)

    # 额外的验证：计算不在 [<configured>, <configured>] 范围内的点数（健壮性检查）
    mask_outside = (data_array < 0) | (data_array > 1)
    count_outside = np.count_nonzero(mask_outside)

    # 整合结果
    total_in_range = count_0a + count_ab + count_b1
    total_elements = data_array.size

    if (total_in_range + count_outside) != total_elements:
        # 这是一个内部一致性检查，理论上不应发生
        print(f"警告：总计数不匹配。总元素：{total_elements}，计算到的：{total_in_range + count_outside}")


    print(f"[0, {a:.4f}){count_0a}")
    print(f"[{a:.4f}, {b:.4f}){count_ab}")
    print(f"[{b:.4f}, 1]{count_b1}")
    print(f"不在 [0, 1] 范围{count_outside}")
    print(f"总计{total_elements}")
    
FILE_PATH=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell5.module.FILE_PATH')
    
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
plot_spectrogram_from_npy(
        npy_filepath=FILE_PATH, 
        title="Example Spectrogram from .npy File"
    )

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import torch
import numpy as np
import matplotlib.pyplot as plt

def simulate_snr_images(
    clean_spectrogram: torch.Tensor,
    target_snr_dbs: list,
    min_val: float = 0.0,
    max_val: float = 1.0,
) -> Dict[float, np.ndarray]:
    '\n    在 PyTorch 中模拟不同信噪比 (SNR) 下的二维频谱图。\n\n    Args:\n        clean_spectrogram: 干净的频谱图（PyTorch Tensor），形状为 [H, W]，\n                           数值应在 [min_val, max_val] 范围内。\n        target_snr_dbs: 目标信噪比列表 (dB)。\n        min_val: 频谱图的最小值（默认为 <configured>）。\n        max_val: 频谱图的最大值（默认为 <configured>）。\n\n    Returns:\n        一个字典，键为 SNR (dB)，值为带噪图像的 NumPy 数组。\n    '
    
    # 确保输入是浮点张量
    S = clean_spectrogram
    H, W = S.shape
    
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # 且我们关注的是峰值部分，通常计算整个图像的平均能量作为信号总功率的代理。
    # 实际应用中，信号功率应只计算信号区域的能量。
    # Historical configuration omitted; provide values through private configuration.
    S_effective = S - min_val
    P_signal = torch.mean(S_effective**2)

    # 结果存储字典
    simulated_images = {}

    for snr_db in target_snr_dbs:
        # Historical configuration omitted; provide values through private configuration.
        # Historical configuration omitted; provide values through private configuration.
        # Historical configuration omitted; provide values through private configuration.
        
        snr_linear = 10**(snr_db / 10.0)
        P_noise_target = P_signal / snr_linear
        
        # Historical configuration omitted; provide values through private configuration.
        # Historical configuration omitted; provide values through private configuration.
        sigma_noise = torch.sqrt(P_noise_target)
        
        # <configured>. 生成高斯白噪声
        noise = torch.randn_like(S) * sigma_noise
        
        # <configured>. 添加噪声
        S_noisy = S + noise
        
        # <configured>. 将带噪结果裁剪/钳位回原始范围 [<configured>, <configured>]
        # 这是工程上的重要步骤，因为噪声可能导致数值超出归一化范围。
        S_clipped = torch.clamp(S_noisy, min=min_val, max=max_val)
        
        # 转换为 NumPy 数组并存储
        simulated_images[snr_db] = S_clipped.cpu().numpy()

    return simulated_images

# --- 演示示例 ---
if __name__ == "__main__":
    
    FILE_PATH=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell6.module.FILE_PATH')
    spec=torch.from_numpy(np.load(FILE_PATH))
    # 目标 SNR 列表 (dB)
    target_snrs = [float('inf'), 20, 10, 5, 0, -5] 
    # 'inf' 表示无噪声 (干净图)

    # 运行模拟
    simulated_results = simulate_snr_images(spec, target_snrs)

    # 绘图展示
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, snr_db in enumerate(target_snrs):
        img_array = simulated_results[snr_db]
        
        # 确保图像颜色映射范围固定，便于比较
        ax = axes[i]
        im = ax.imshow(img_array, aspect='auto', cmap='magma', vmin=0, vmax=1)
        
        title = f"SNR: {snr_db} dB (Clean)" if snr_db == float('inf') else f"SNR: {snr_db} dB"
        ax.set_title(title, fontsize=14)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#将.parquet文件中数据提取为.jpg（.jpg格式原始数据）
import pandas as pd
import os
from PIL import Image
from io import BytesIO

# --- 定义路径和变量 ---
# 假设您的 Parquet 文件都位于这个目录下（这个路径需要您根据实际情况修改）
PARQUET_DIR = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell7.module.PARQUET_DIR') 
# 目标输出文件夹
OUTPUT_ROOT_DIR = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell7.module.OUTPUT_ROOT_DIR')

def process_parquet_file(file_path, output_root_dir):
    """
    加载单个 Parquet 文件，提取图像和标签，并按标签保存图像文件。
    """
    try:
        # <configured>. 加载 Parquet 文件到 Pandas DataFrame
        # 注意：使用 engine='pyarrow' 确保兼容性
        df = pd.read_parquet(file_path, engine='pyarrow')
    except Exception as e:
        print(f"!!! 无法加载文件 {file_path}: {e}")
        return

    file_name_prefix = os.path.basename(file_path).split('.')[0]
    
    print(f"正在处理文件: {file_name_prefix}.parquet ({len(df)} 个样本)")

    # <configured>. 遍历 DataFrame 的每一行
    for index, row in df.iterrows():
        # 提取标签
        label = str(row['label']) 
        
        # 提取图像字典和字节数据
        image_dict = row['image']
        image_bytes = image_dict.get('bytes') # 从字典中安全地获取 'bytes' 键的值

        if image_bytes is None:
            print(f"跳过样本 {index}：'image' 列中缺少 'bytes' 键。")
            continue

        # <configured>. 构造输出路径
        # 目标标签文件夹路径
        label_dir = os.path.join(output_root_dir, label)
        # 确保标签文件夹存在
        os.makedirs(label_dir, exist_ok=True)
        
        # 构造保存文件名：使用 parquet 文件名 + 样本索引
        output_filename = f"{file_name_prefix}_{index:05d}.jpg"
        output_path = os.path.join(label_dir, output_filename)

        # Historical configuration omitted; provide values through private configuration.
        # 因为我们知道字节数据已经是完整的 JPEG 格式，无需 PIL 解码和重新编码
        try:
            with open(output_path, 'wb') as f:
                f.write(image_bytes)
            
            # 如果需要验证文件是否损坏，可以取消注释下面的 PIL 检查
            # Image.open(BytesIO(image_bytes)).save(output_path) 
            
        except Exception as e:
            print(f"!!! 无法将样本 {index} 保存到 {output_path}: {e}")


def batch_process_parquet_files(parquet_dir, output_root_dir):
    """
    遍历指定目录下的所有 Parquet 文件，并逐个处理。
    """
    # 查找所有 .parquet 文件
    parquet_files = [f for f in os.listdir(parquet_dir) if f.endswith(_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell7.batch_process_parquet_files.path'))]
    parquet_files.sort() # 按名称排序，确保按 <configured>, <configured>... 顺序处理

    if not parquet_files:
        print(f"警告：在目录 {parquet_dir} 中未找到任何 .parquet 文件。")
        return

    print(f"总共找到 {len(parquet_files)} 个 .parquet 文件准备处理。")

    for file_name in parquet_files:
        file_path = os.path.join(parquet_dir, file_name)
        process_parquet_file(file_path, output_root_dir)
    
    print("\n--- 批量处理完成 ---")
    
    
# --- 运行批量处理 ---
batch_process_parquet_files(PARQUET_DIR, OUTPUT_ROOT_DIR)

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import torch
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.decomposition import IncrementalPCA
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm

class SingleSampleNPYDataset(torch.utils.data.Dataset):
    def __init__(self, npy_dir):
        self.npy_dir = Path(npy_dir)
        self.file_list = list(self.npy_dir.glob(_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.SingleSampleNPYDataset.__init__.path')))
        print(f"找到 {len(self.file_list)} 个样本文件")
        
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        data = np.load(self.file_list[idx])
        return data.astype(np.float32)

def incremental_pca_high_dim(npy_dir, n_components=None, batch_size=None):
    """
    使用增量PCA处理高维数据
    """
    n_components = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell8.incremental_pca_high_dim.n_components', n_components)
    batch_size = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell8.incremental_pca_high_dim.batch_size', batch_size)
    dataset = SingleSampleNPYDataset(npy_dir)
    
    # 获取数据维度
    sample_data = dataset[0]
    n_features = sample_data.shape[0] * sample_data.shape[1]  # <configured> * <configured>
    
    print(f"数据维度: {sample_data.shape} -> {n_features} 特征")
    print(f"使用增量PCA，批次大小: {batch_size}")
    
    # 初始化增量PCA
    ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)
    
    # 分批拟合
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.incremental_pca_high_dim.num_workers')
    )
    
    for i, batch in enumerate(tqdm(dataloader, desc="增量PCA拟合")):
        # 将批次数据展平
        if isinstance(batch, torch.Tensor):
            batch = batch.numpy()
        
        batch_flat = batch.reshape(batch.shape[0], -1)
        ipca.partial_fit(batch_flat)
        
        if i % 10 == 0:
            print(f"已处理 {i * batch_size}/{len(dataset)} 样本")
    
    print(f"解释方差比: {ipca.explained_variance_ratio_}")
    print(f"累计解释方差: {ipca.explained_variance_ratio_.sum():.4f}")
    
    return ipca

def transform_with_ipca(npy_dir, ipca, batch_size=None, sample_fraction=None):
    """
    使用训练好的增量PCA转换数据
    """
    batch_size = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell8.transform_with_ipca.batch_size', batch_size)
    sample_fraction = _cfg_resolve('GPN_UAV_data/extract&draw_picture.ipynb.cell8.transform_with_ipca.sample_fraction', sample_fraction)
    dataset = SingleSampleNPYDataset(npy_dir)
    
    # 采样
    if sample_fraction < 1.0:
        sample_size = int(len(dataset) * sample_fraction)
        indices = np.random.choice(len(dataset), sample_size, replace=False)
        sampler = torch.utils.data.SubsetRandomSampler(indices)
        dataloader = torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, sampler=sampler
        )
    else:
        dataloader = torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, shuffle=False
        )
    
    all_transformed = []
    
    for batch in tqdm(dataloader, desc="转换数据"):
        if isinstance(batch, torch.Tensor):
            batch = batch.numpy()
        
        batch_flat = batch.reshape(batch.shape[0], -1)
        transformed = ipca.transform(batch_flat)
        all_transformed.append(transformed)
    
    return np.vstack(all_transformed)


def main_high_dim():
    """
    高维数据PCA主函数
    """
    npy_directory = _cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.npy_directory')
    
    print("=== 高维数据PCA分析 ===")
    print('数据特征维度: <configured> × <configured> = <configured>,<configured>')
    
    # 方法<configured>：增量PCA（最推荐）
    print("\n方法1: 使用增量PCA")
    ipca = incremental_pca_high_dim(npy_directory, n_components=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.n_components'), batch_size=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.batch_size'))
    transformed_data = transform_with_ipca(npy_directory, ipca, batch_size=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.batch_size__2'), sample_fraction=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.sample_fraction'))
    
    # 可视化
    plt.figure(figsize=(10, 8))
    plt.scatter(transformed_data[:, 0], transformed_data[:, 1], alpha=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.alpha'), s=30)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(f'PCA Visualization - {len(transformed_data)} samples')
    plt.grid(True, alpha=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.alpha__2'))
    plt.savefig('pca_high_dim.png', dpi=_cfg_require('GPN_UAV_data/extract&draw_picture.ipynb.cell8.main_high_dim.size_or_budget'), bbox_inches='tight')
    plt.show()
    
    # 保存结果
    # np.save('transformed_data_high_dim.npy', transformed_data)
    
    print(f"转换后数据形状: {transformed_data.shape}")
    print("分析完成！")

if __name__ == "__main__":
    main_high_dim()